# Bedding × Dinomaly — training tutorial

Trains the 6-channel ALL6 Dinomaly anomaly-detection pipeline on the bedding dataset, end to end and **self-contained** — the data download, the cu3s→NPZ conversion, the pipeline construction, and the training loop are all inline (no external scripts). The result is a saved pipeline you can run with the companion `bedding_all6_inference_tutorial.ipynb` (set `BEDDING_PIPELINE_SOURCE=local`).

Covers:
1. Dataset overview (6 bands, 252 frames).
2. Get + convert the data (HuggingFace cu3s → cropped NPZ).
3. Build the 6-channel pipeline node-by-node.
4. Train (statistical init + gradient training); `MAX_EPOCHS` knob.
5. Save the trained pipeline (YAML + .pt).
6. Use your trained pipeline in the inference notebook.

> **Prerequisites**
>
> 1. Run from inside the `cuvis-ai-dinomaly` repo (paths auto-resolve relative to it).
> 2. Activate the repo env with the cuvis SDK on the path: `CUVIS=/lib/cuvis` (or your SDK dir).
> 3. High-level `cuvis-ai` must be importable (the pipeline uses `cuvis_ai.node.*` built-ins). `pip install cuvis-ai` if needed.
> 4. **Data comes from HuggingFace by default** (`cubert-gmbh/X4_SWIR_Industrial_Foreign_Object_Detection_Bedding`). Set `BEDDING_DATA_SOURCE=local` to use a local mount instead.
> 5. GPU strongly recommended — the encoder is DINOv2 ViT-B/14.

## 1 · Dataset overview

- **Source**: published on HuggingFace at `cubert-gmbh/X4_SWIR_Industrial_Foreign_Object_Detection_Bedding` as cu3s session files, one per frame.
- **Channels**: 6 bands at 450 / 550 / 625 nm (VIS) and 1050 / 1200 / 1450 nm (SWIR).
- **Spatial resolution**: native **2400 × 4900**; the pretrained pipeline trains on a center-crop to **1800 × 4300** (aspect 2.39:1). The converter below applies that crop.
- **Scene**: a sawdust-filled tray with foreign objects — PLA cubes (multiple sizes + colours), leaves, plastic foil, PET, POMC, water, alcohol.
- **Splits**: 193 train + 59 val (no test). Train is all-normal (unsupervised setup); val mixes normal `_ok_ok_` frames with anomalous frames.
- **Anomaly classes**: 23 (see `comparisons/per_class_auroc_matrix.md`).

In [ ]:
%matplotlib inline
import sys, csv
from pathlib import Path

import numpy as np
import torch
import pytorch_lightning as pl
from PIL import Image

sys.path.insert(0, str(Path('.').resolve()))
import utils
from utils import (
    resolve_default_config, load_bedding_splits, load_bedding_cube,
    load_bedding_mask_path, center_crop_to_training,
    BEDDING_ALL6_NM, BEDDING_ALL6_LABELS, BEDDING_HF_CACHE,
)

config = resolve_default_config()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:        ', DEVICE)
print('Dataset source:', config['data_source'])
print('Target λ order:', BEDDING_ALL6_NM)

# --- Tutorial knobs -------------------------------------------------------
MAX_EPOCHS   = 1     # bump to ~20 for a real run (the published model is 20 epochs)
IMAGE_SIZE   = 448   # square squash side (multiple of 14); 672 for the high-res pilot
CONVERT_LIMIT = 0    # 0 = convert all 252 frames; set e.g. 16 for a fast dry-run
OUTPUT_DIR   = Path('outputs/trained_run')   # inference notebook reads this when BEDDING_PIPELINE_SOURCE=local
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2 · Get + convert the data (HuggingFace cu3s → cropped NPZ)

`train_bedding_all6` trains from pre-decoded **NPZ** frames (one cube per file). This cell builds them inline: for every frame in the dataset `splits.csv` it loads the cu3s cube (downloaded from HuggingFace + **center-cropped 2400×4900 → 1800×4300** to match the model's training field-of-view), pairs it with its GT mask, and writes a compressed NPZ + a splits CSV that the datamodule consumes. NPZ is also far faster per-epoch than decoding cu3s every step.

> Conversion runs once and caches under `~/.cache/cuvis_bedding/npz`. All 252 frames take a few minutes; set `CONVERT_LIMIT` above for a quick dry-run.

In [ ]:
if config['data_source'] == 'local':
    TRAIN_SPLITS_CSV = config['splits_csv']
    print('local mode — using existing NPZ splits:', TRAIN_SPLITS_CSV)
else:
    NPZ_OUT = BEDDING_HF_CACHE / 'npz'
    TRAIN_SPLITS_CSV = NPZ_OUT / 'bedding_splits_npz.csv'
    if not TRAIN_SPLITS_CSV.is_file():
        splits = load_bedding_splits()                 # HF splits.csv (split, stem, …)
        rows = []
        per_split_count = {}
        for _, r in splits.iterrows():
            split, stem = r['split'], r['stem']
            if CONVERT_LIMIT and per_split_count.get(split, 0) >= CONVERT_LIMIT:
                continue
            per_split_count[split] = per_split_count.get(split, 0) + 1
            cube, wl = load_bedding_cube(stem, split=split)          # cropped (1800,4300,6) float32
            mask_path = load_bedding_mask_path(stem)
            if mask_path is not None:
                m = center_crop_to_training(np.asarray(Image.open(mask_path)))
                mask = (m > 0).astype(np.int32)
                class_mask = m.astype(np.uint8)
            else:                                                     # normal frame → empty mask
                mask = np.zeros(cube.shape[:2], dtype=np.int32)
                class_mask = np.zeros(cube.shape[:2], dtype=np.uint8)
            out = NPZ_OUT / split / f'{stem}.npz'
            out.parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(out, cube=cube, wavelengths=wl.astype(np.int32),
                                mask=mask, class_mask=class_mask)
            rows.append({'npz_path': str(out), 'cu3s_path': '', 'annotation_json': '',
                         'image_id': 0, 'mask_path': '', 'split': split})
        with open(TRAIN_SPLITS_CSV, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=['npz_path','cu3s_path','annotation_json','image_id','mask_path','split'])
            w.writeheader(); w.writerows(rows)
        print(f'converted {len(rows)} frames → {TRAIN_SPLITS_CSV}')
    else:
        print('NPZ already present →', TRAIN_SPLITS_CSV)
print('Train splits CSV:', TRAIN_SPLITS_CSV)

## 3 · Build the 6-channel pipeline

The graph, built node-by-node:

```
LentilsAnomalyDataNode            # serves NPZ cube + wavelengths + mask
 → MinMaxNormalizer               # running [0,1] scaling
 → FixedWavelengthSelector (upstream)  # pick + reorder the 6 target bands (norm_mode=per_frame)
 → DinomalyDetector(input_channels=6)   # DINOv2 patch-embed inflated 3→6
 → DinomalyTrainLossBridge        # training loss
 → QuantileBinaryDecider → AnomalyDetectionMetrics → TensorBoard
 + AnomalyAUROCMetrics            # epoch-end pixel/image AUROC
```

`input_channels=6` triggers the duplicate-and-halve patch-embed inflation (the inflated conv is a fixed activation-parity stem — it stays frozen (anomalib runs the encoder under no_grad, so it gets no gradient)). The selector stacks `(625, 550, 450, 1450, 1200, 1050) nm` — descending λ within each VIS/SWIR triplet so the inflated conv sees matched per-slot statistics.

> **Data module.** `MultiNpzDataModule` (`npz_multi`) ships in the **cuvis-ai-dataloader** plugin (v0.3.0+), not in this repo. It reads one cube per `.npz` listed in a splits CSV (`split, npz_path, image_id`; extra columns ignored) and emits `{cube, mask, wavelengths, mesu_index}`. It honors `pin_memory` / `persistent_workers` / `worker_multiprocessing_context`. See the [cuvis-ai-dataloader README](https://github.com/cubert-hyperspectral/cuvis-ai-dataloader#npz-npz_multi) for the full usage note.

In [ ]:
from cuvis_ai.node.data import LentilsAnomalyDataNode
from cuvis_ai.node.normalization import MinMaxNormalizer
from cuvis_ai.node.metrics import AnomalyDetectionMetrics
from cuvis_ai.node.monitor import TensorBoardMonitorNode
from cuvis_ai.deciders.binary_decider import QuantileBinaryDecider
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_dataloader.data import MultiNpzDataModule
from cuvis_ai_dinomaly.node.dinomaly_detector import DinomalyDetector
from cuvis_ai.node.channel_selector import FixedWavelengthSelector
from cuvis_ai_dinomaly.node.dinomaly_train_loss_bridge import DinomalyTrainLossBridge
from cuvis_ai_dinomaly.node.auroc_metrics import AnomalyAUROCMetrics

pl.seed_everything(42, workers=True)

datamodule = MultiNpzDataModule(splits_csv=str(TRAIN_SPLITS_CSV), batch_size=1, num_workers=0)
datamodule.setup(stage='fit')

pipeline = CuvisPipeline('dinomaly_bedding_all6')
data_node  = LentilsAnomalyDataNode(normal_class_ids=[0])
normalizer = MinMaxNormalizer(eps=1e-6, use_running_stats=True, max_initialization_frames=None)
selector   = FixedWavelengthSelector(target_wavelengths=BEDDING_ALL6_NM,
                                     normalize_output=False, norm_mode='per_frame',
                                     name='hyperspectral_selector')
dinomaly   = DinomalyDetector(
    encoder_name='dinov2reg_vit_base_14', bottleneck_dropout=0.2, decoder_depth=8,
    image_size=IMAGE_SIZE, crop_size=IMAGE_SIZE, use_center_crop=False,
    input_channels=6, name='dinomaly_detector')
loss_bridge  = DinomalyTrainLossBridge(weight=1.0, name='dinomaly_train_loss')
decider      = QuantileBinaryDecider(quantile=0.995, name='decider')
metrics_node = AnomalyDetectionMetrics(name='metrics_anomaly')
auroc_node   = AnomalyAUROCMetrics(name='metrics_auroc')
tb           = TensorBoardMonitorNode(output_dir=str(OUTPUT_DIR / 'tensorboard'), run_name=pipeline.name)

pipeline.connect(
    (data_node.outputs.cube,            normalizer.data),
    (normalizer.normalized,             selector.cube),
    (data_node.outputs.wavelengths,     selector.wavelengths),
    (selector.rgb_image,                dinomaly.rgb_image),
    (dinomaly.outputs.training_loss,    loss_bridge.raw_loss),
    (dinomaly.outputs.scores,           decider.logits),
    (dinomaly.outputs.scores,           metrics_node.logits),
    (decider.decisions,                 metrics_node.decisions),
    (data_node.outputs.mask,            metrics_node.targets),
    (metrics_node.metrics,              tb.metrics),
    (dinomaly.outputs.scores,           auroc_node.scores),
    (data_node.outputs.mask,            auroc_node.targets),
    (dinomaly.outputs.anomaly_score,    auroc_node.anomaly_score),
)
print('Pipeline built:', [n.name for n in pipeline.nodes])

## 4 · Train

Two phases, mirroring the production script:
1. **Statistical initialization** — fits the `MinMaxNormalizer` running bounds.
2. **Gradient training** — unfreezes the Dinomaly bottleneck + decoder (the inflated patch-embed stays a fixed stem) and runs the Lightning loop, with the AUROC node streaming pixel/image AUROC through the trainer's per-batch metric logging (TensorBoard).

`MAX_EPOCHS` (set above) controls the run length — `1` is a quick smoke; bump to ~20 for a real model.

In [ ]:
from cuvis_ai_core.training import GradientTrainer, StatisticalTrainer
from cuvis_ai_core.training.config import create_callbacks_from_config
from cuvis_ai_schemas.training import (
    CallbacksConfig, ModelCheckpointConfig, OptimizerConfig, TrainerConfig,
)

trainer_cfg = TrainerConfig(
    max_epochs=MAX_EPOCHS, accelerator='auto', devices=1,
    default_root_dir=str(OUTPUT_DIR), precision='32-true',
    enable_progress_bar=True, enable_checkpointing=True, log_every_n_steps=10,
    check_val_every_n_epoch=1, gradient_clip_val=0.1,
    callbacks=CallbacksConfig(checkpoint=ModelCheckpointConfig(
        dirpath=str(OUTPUT_DIR / 'checkpoints'), monitor='metrics_anomaly/iou',
        mode='max', save_top_k=1, save_last=True, filename='{epoch:02d}')),
)
optimizer_cfg = OptimizerConfig(name='adamw', lr=2e-4, weight_decay=1e-4, betas=[0.9, 0.999])

if normalizer.requires_initial_fit:
    print('Phase 1: statistical initialization (MinMaxNormalizer)…')
    StatisticalTrainer(pipeline=pipeline, datamodule=datamodule).fit()

pipeline.unfreeze_nodes_by_name(['dinomaly_detector'])
pipeline.to(DEVICE)
n_train = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in pipeline.parameters())
print(f'Trainable: {n_train:,} / {n_total:,} ({100*n_train/max(n_total,1):.2f}%)')

callbacks = list(create_callbacks_from_config(trainer_cfg.callbacks)) if trainer_cfg.callbacks else []

print(f'Phase 2: gradient training for {MAX_EPOCHS} epoch(s)…')
grad_trainer = GradientTrainer(
    pipeline=pipeline, datamodule=datamodule,
    loss_nodes=[loss_bridge], metric_nodes=[metrics_node, auroc_node],
    trainer_config=trainer_cfg, optimizer_config=optimizer_cfg,
    monitors=[tb], callbacks=callbacks,
)
grad_trainer.fit()
print('Training done.')

## 5 · Save the trained pipeline

`save_to_file` writes the pipeline YAML + a co-located `.pt` of all node weights. These two files are everything the inference notebook needs.

In [ ]:
from cuvis_ai_schemas.pipeline import PipelineMetadata

results_dir = OUTPUT_DIR / 'trained_models'
results_dir.mkdir(parents=True, exist_ok=True)
pipeline_path = results_dir / f'{pipeline.name}.yaml'
pipeline.save_to_file(
    str(pipeline_path),
    metadata=PipelineMetadata(
        name=pipeline.name,
        description='Dinomaly on bedding hyperspectral NPZ, 6-band selector '
                    '(625/550/450/1450/1200/1050 nm), input_channels=6 patch-embed inflation.',
        tags=['dinomaly', 'anomalib', 'bedding', 'all6', 'hyperspectral'],
        author='cuvis.ai'),
)
print('Saved:', pipeline_path)
print('Weights:', pipeline_path.with_suffix('.pt'),
      f"({pipeline_path.with_suffix('.pt').stat().st_size/1e6:.0f} MB)" if pipeline_path.with_suffix('.pt').is_file() else '(missing)')
print('\n'.join(pipeline_path.read_text().splitlines()[:25]))

## 6 · Use your trained pipeline in the inference notebook

Point the inference tutorial at what you just trained instead of the published HF model:

```bash
export BEDDING_PIPELINE_SOURCE=local
export BEDDING_PIPELINE_DIR=$(pwd)/outputs/trained_run/trained_models
jupyter lab   # then run bedding_all6_inference_tutorial.ipynb
```

The inference notebook's `resolve_pipeline()` picks up the single `*.yaml` (+ its `.pt`) from that directory. Everything else — the cu3s loading, crop, scoring, speedup recipe — is identical.

For aspect-preserving training (preserve the 2.39:1 aspect instead of square squash), set `IMAGE_SIZE` above to a tuple-driven run by passing `image_size=(434, 1036)` to `DinomalyDetector` (both dims multiples of 14); the rectangular-input monkey-patch activates automatically when H ≠ W.